> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Additive prediction and gdec solver comparison

VIGOR 2.12.1. This portable example needs no R environment; the rrBLUP comparator is in examples/real26.


In [ ]:
from pathlib import Path
import sys,os,importlib.util
import numpy as np,pandas as pd,vigor
assert vigor.__version__ == "2.12.1"
from IPython.display import display
cfg=vigor.simulation_scenario("complex");cfg.update(heritability=.55,compute_grms=True)
sim=vigor.simulate_genomic_dataset(n_samples=240,n_variants=500,n_traits=6,seed=42,
    genotype_kwargs={"n_chromosomes":3},trait_kwargs=cfg)
X=sim.genotypes;Y=sim.phenotypes;K=sim.grms["additive"].to_numpy(float)
rng=np.random.default_rng(42);train=Y.mask(rng.random(Y.shape)<.15);train.iloc[:48]=np.nan


In [ ]:
common=dict(rank_G="full",rank_E="full",solver="lbfgs",analytic_se=False,n_boot=0,progress=False,blas_threads=2,random_state=42)
fit=vigor.genetic_varcov_PSD(K,train,**common)
blup,aux=vigor.mvBLUP(train,K,fit,accuracy="exact",return_aux=True,preconditioner="diag",blas_threads=2)
predict=vigor.mvPredict(train,K,fit,accuracy="exact",preconditioner="diag",blas_threads=2)
rows=[]
mask=train.isna().to_numpy()&Y.notna().to_numpy()
for name,value in [("mvBLUP",blup),("mvPredict",predict["phenotype_prediction"])]:
    y=Y.to_numpy()[mask];p=np.asarray(value)[mask]
    rows.append({"model":name,"predictive_r2":1-np.sum((y-p)**2)/np.sum((y-y.mean())**2),"corr2":np.corrcoef(y,p)[0,1]**2,"rmse":np.sqrt(np.mean((y-p)**2))})
display(pd.DataFrame(rows))
path=Path(vigor.__file__).resolve().parent.parent/"comparators/genome_decomposition_v3_1_3.py"
if path.exists():
    spec=importlib.util.spec_from_file_location("gdec313",path);gdec=importlib.util.module_from_spec(spec);spec.loader.exec_module(gdec)
    G=fit.get("G_prediction",fit["G"]);E=fit.get("E_prediction",fit["E"])
    old=gdec.mvBLUP(train,K,G,E,missing="exact",preconditioner="diag",blas_threads=2)
    new=vigor.mvBLUP(train,K,G,E,accuracy="exact",preconditioner="diag",blas_threads=2)
    print("Shared covariance max absolute difference:",float(np.max(np.abs(np.asarray(old)-np.asarray(new)))))
else:print("gdec comparator unavailable")
